# Claims data

For paid and case estiamtes, we have received diagonal data from the client.
For paid claims, we also have 2024 and 2023 paid claims, which we may be able to check NMG's previous diagonals, otherwise its just the 2025 diagonal to be generated. That is why we need the database of previous triangles.


In [2]:
from pathlib import Path
import sys
import pandas as pd


In [2]:
import os, pandas as pd, chainladder as cl

db_prev_data_path =  REPO_ROOT / 'data' / 'db' / 'database.xlsx'

#prev_claims = pd.read_excel(db_prev_data_path, sheet_name='triangle')


In [3]:

# This is the tab with historic incremental, gross paid claims
paid = pd.read_excel(db_prev_data_path, sheet_name='paid')

#prev_reported = prev_claims[prev_claims['data_type']=='Reported']

## Current valuation date data


In [4]:
#Generally Sun Re provide case estimate and paid data in the same spreadsheet.

claims_data_path = '/Users/sivaramankumar/Library/CloudStorage/OneDrive-Personal/_Ryskk/Clients/Sun Re/2026/Q2/manipulated_data/LFIC and LAE Sun RE 2Q 2026.xlsx'


# Paid Data
### Bring in latest quarters/ year worth of data...


Before you do this, go into the file and do manual amendments. As the paid claims list is usually quite small, its easier to do at source than in the code.

Notes:
1. Amend date formats (sometimes separators are . and theres a mix of dd/mm and mm/dd.. generally flip to dd/mm, where its ambiguous leave as is, given that the year is most important here) - mainly for underwriting year
2. For date of payment, shouldn't matter too much as we are doing AY/DY, so can default to the valuation year
3. In the excel data set that they share, they don't have an accident date for the bordereaux business. So for this I would you the bordereaux date
4. Add columns (because its easier to do in excel than here, for ay,uwy, dy and payment)
5. Put notes in the manipulated version of the sheet, so that can track manual adjustments
6. Generally LoB in Paid data is clean (compared to case estimates, but worth keeping an eye on and checking)

In [5]:
paid_df = pd.read_excel(claims_data_path, sheet_name='Claims paid 2Q')

# Case estimates

For case estimates, as the data structure is bigger and more complext, we can read directly here, however some manipulations are easier to do in excel... keep notes on these in the file (as the paid claims)

In [11]:
os_df = pd.read_excel(claims_data_path, sheet_name='Loss reserves')

In [12]:
#assumes contract number pattern is like "SUN-CYB-25-0113" where 25 is the year of underwriting
year = os_df["Contract No"].str.extract(r"-(\d{2})-")[0].astype("float")

dol = pd.to_datetime(os_df["Date of Loss"], errors="coerce")
nol = pd.to_datetime(os_df["Notice of loss"], errors="coerce")


#if error/ empty contract no then take dol othewise dol
os_df["uwy"] = year.where(year.between(0, 30)).add(2000).fillna(dol.dt.year).fillna(nol.dt.year)



In [ ]:
import re
from datetime import datetime

current_year = datetime.now().year

def extract_year(val):
    matches = re.findall(r'\b(20\d{2})\b', str(val))
    valid = [int(y) for y in matches if current_year - 10 <= int(y) <= current_year]
    return valid[0] if valid else None

os_df['orig_year'] = os_df['Contract No'].apply(extract_year)

In [9]:
# For ay, use date of loss where available, otherwise use notice of loss


os_df["ay"] = dol.dt.year.fillna(nol.dt.year)

In [9]:
# For where there are no UWY in the data (awaiting clarification from client), we set the uwy to the date of loss 

os_df["uwy"] = os_df["uwy"].fillna(os_df["ay"])


In [10]:
# APRIL 2026 Note: Will need to come back to this and think about a better mapping and align with the paid data and written premiums as well!

lob_group_map = {
    'Marine: H&M': 'Marine',
    'Marine Various': 'Marine',
    'Marine: Yacht': 'Marine',
    'Marine: P&I': 'Marine',
    'Marine: Cargo': 'Marine',
    'Marine: Rolling Stock': 'Marine',
    'Marine: Builder\'s': 'Marine',
    'Marine: Voyage': 'Marine',
    'Marine': 'Marine',

    'Aviation: Hull': 'Aviation',
    'Aviation': 'Aviation',

    'Financial: Banker’s blanket bonds': 'Financial',
    'Financial: Guarantees & Bonds': 'Financial',
    'Financial: Political Risks': 'Financial',
    'Financial: Surety': 'Financial',
    'Financial': 'Financial',

    'Personal: Health': 'Personal',
    'Personal: Travel': 'Personal',
    'Personal': 'Personal',

    'P&C: Industrial': 'Property',
    'P&C: Energy': 'Property',
    'P&C: Commercial': 'Property',
    'P&C: Engineering, CAR/EAR, CECR': 'Property',
    'P&C': 'Property',

    'Liability: Medical Malpractice': 'Liability',
    'Liability: D&O': 'Liability',
    'Liability: General & Miscellaneous': 'Liability',
    'Liability: Cyber': 'Liability',
    'Liability: Professional indemnity': 'Liability',
    'Liability': 'Liability',

    'Treaty: Quota Share': 'Treaty',
    'Treaty: Excess of Loss': 'Treaty',
    'Treaty': 'Treaty',
    'Motor': 'Personal',

    'Other': 'Misc'
}

In [11]:
os_df['Line'].unique()

array(['Liability', 'Marine: H&M', 'Marine Various', 'Marine',
       'Financial: Banker’s blanket bonds',
       'Liability: Medical Malpractice', 'Financial', 'P&C: Industrial',
       'Aviation: Hull', 'Aviation', 'Marine: Yacht', 'Marine: P&I',
       'Marine: Cargo', 'Personal: Health', 'Treaty: Quota Share',
       'Liability: D&O', 'Treaty: Excess of Loss', 'P&C: Energy', 'P&C',
       'Liability: General & Miscellaneous', 'Treaty', 'Personal',
       'P&C: Commercial', 'Financial: Guarantees & Bonds', 'Other',
       'Marine: Rolling Stock', 'Motor', "Marine: Builder's",
       'P&C: Engineering, CAR/EAR, CECR', 'Liability: Cyber',
       'Financial: Political Risks', 'Marine: Voyage',
       'Liability: Professional indemnity'], dtype=object)

In [12]:
os_df['lob'] = os_df['Line'].map(lob_group_map)

In [13]:
os_df['dy'] = val_year

From client: Loss+ LAE expenses is claims + expenses. But these should already be covered in the paid claims data.
We need to then allow for the LAE expenses in the paid claims (the case estimate share is already in the case estimate data)


In [14]:
os_df['case_estimates'] = os_df['Liability for incurred claims, Amount Outstanding,USD'] + pd.to_numeric(os_df['Loss adjustment expenses, Amount Outstanding,USD'], errors='coerce').fillna(0)
os_df['recoveries'] = os_df['Gross Retrocession Recoverables, Amount,USD'] - os_df['Reinstatement premium payable, USD']

Bring in the database OS's and apend new data

In [15]:
os_df['val_date'] = pd.to_datetime(val_date)
os_df['data_type'] = 'Case Outstanding'
os_df['gross_net_re'] = 'Gross'
os_df['cum_inc'] = 'cumulative'
os_df['origin_period'] = os_df['uwy']
os_df['dev_period'] = os_df['dy']
os_df['accident_period'] = os_df['ay']
os_df['value'] = os_df['case_estimates']



In [16]:
os_df[['val_date', 'data_type', 'lob','gross_net_re', 'cum_inc','origin_period', 'dev_period', 'value']].to_csv('C:\\Github\\selecta_valuation\\data\\db\\case_outstanding_2026Q1.csv', index=False)



### Retrocession

In [17]:
#For calculatimg recoveries. we are including reinstatement premiums payable

#yet to be paid by the reinsurer, so basically adjustment to case estimates...
os_df['case_estimate_recoveries'] = os_df['Gross Retrocession Recoverables, Amount,USD'] -  os_df['Reinstatement premium payable, USD']

#already been settled by the reinsurer 
os_df['paid_recovered'] = os_df['Gross Retrocession Recovered, Amount ,USD']


In [18]:
#Use this for net of re case estimates, we can use this or just recoveries for IFRS17 LIC/LRC

os_df['net_ri_case_estimates'] = os_df['case_estimates'] - os_df['case_estimate_recoveries']

# Gross to net ratios

Can see that there are negative net ri case estimates - from closed policies, combination of reinstatement premiums and retrocession recoveries.. so just delays..


Probably need to treat the gross-to-net ratios a bit differently to the underlying data.... as we have losses paid and lae in the dataset,

We theoretically should be able to get a view of gross-to-net on reported (though limited to those claims in this dataset, which I can't tell is all the claims ever - highly unlikely or all the claims that had activitiy in 2025). Though these may not reconcile with the base paid and case estimates data... but we only need this for gross to net and it looks like its only going to affect Marine, as there's no cover in other LoBs

Therefore, for simplicity, net to gross ratios for os and reported calculated below

In [19]:
os_df.columns

Index(['Contract No', 'Original No', 'Claim No', 'Type contract',
       'Who is involved', 'Line', 'AM Best', 'Notice of loss', 'Date of Loss',
       'Latest reestimation date', 'Reestimation reminder', 'Underwriter',
       'Thread', 'Brief description', 'Source', 'Claimant', 'Status',
       'Liability for incurred claims, Amount Outstanding,CCY',
       'Liability for incurred claims,CCY',
       'Liability for incurred claims,FX rate',
       'Liability for incurred claims, Amount Outstanding,USD',
       'Liability for incurred claims+LAE, Amount Paid&Checked,USD',
       'Loss adjustment expenses, Amount Outstanding,CCY',
       'Loss adjustment expenses,CCY', 'Loss adjustment expenses,FX rate',
       'Loss adjustment expenses, Amount Outstanding,USD',
       'Gross Retrocession Recoverables, Amount,CCY',
       'Gross Retrocession Recoverables,CCY',
       'Gross Retrocession Recoverables,FX rate',
       'Gross Retrocession Recoverables, Amount,USD',
       'Gross Retrocessi

In [20]:
#os g2n
gross_to_net_os_ratios = (os_df.groupby(['uwy', 'lob'])['net_ri_case_estimates'].sum()/os_df.groupby(['uwy', 'lob'])['case_estimates'].sum()).reset_index().fillna(0)

In [21]:
gross_to_net_os_ratios

,uwy,lob,0
0,2016.0,Property,0.000000
1,2017.0,Financial,0.000000
2,2017.0,Marine,0.000000
3,2017.0,Property,0.000000
4,2018.0,Marine,0.000000
5,2018.0,Misc,0.000000
6,2018.0,Property,0.000000
7,2019.0,Financial,1.000000
8,2019.0,Marine,1.000000
9,2019.0,Personal,0.000000


These negatives are because there are no case estimates, so these are the delays... I think we just pass this through to the triangles/ results and not use them for case estimates

In [22]:
gross_to_net_reported_ratio_df = os_df.copy()

#just for testing purposes, look at gross to net ratios by claim..
gross_to_net_reported_ratio_df


,Contract No,Original No,Claim No,Type contract,Who is involved,Line,AM Best,Notice of loss,Date of Loss,Latest reestimation date,...,data_type,gross_net_re,cum_inc,origin_period,dev_period,accident_period,value,case_estimate_recoveries,paid_recovered,net_ri_case_estimates
0,SCR-LBT-20-0101,B1230PC06047A20,13451,reinsurance,Yes,Liability,Medical Prof Liab (Claims Made),2022-08-29,2021-11-10,2025-11-10,...,Case Outstanding,Gross,cumulative,2020.0,2026,2021.0,245512.55,0,0,245512.55
1,SUN-MAR-23-0229,3384D21E-C54E-4A6F-9E83-D28EFAF886D4,15523,direct,selecta,Marine: H&M,"Marine, Aviation & Transport",2023-05-10,2023-04-29,2025-08-13,...,Case Outstanding,Gross,cumulative,2023.0,2026,2023.0,0.00,0,0,0.00
2,SUN-MAR-22-0284,3384D21E-C54E-4A6F-9E83-D28EFAF886D4,15139,direct,sunshine_selecta,Marine Various,"Marine, Aviation & Transport",2023-03-20,2023-02-27,2025-10-16,...,Case Outstanding,Gross,cumulative,2022.0,2026,2023.0,0.00,0,0,0.00
3,SUN-MAR-22-0284,3384D21E-C54E-4A6F-9E83-D28EFAF886D4,14239,direct,sunshine_selecta,Marine: H&M,"Marine, Aviation & Transport",2022-11-30,2022-11-23,2025-01-17,...,Case Outstanding,Gross,cumulative,2022.0,2026,2022.0,0.00,0,0,0.00
4,SUN-MHU-21-0103,20210066/20210073/2020,12804,reinsurance,Yes,Marine,"Marine, Aviation & Transport",2022-06-15,2021-07-27,2026-02-16,...,Case Outstanding,Gross,cumulative,2021.0,2026,2021.0,0.00,0,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1533,-,-,"16: 1911.020, 1911.021, 1912.027",reinsurance,-,Marine,"Marine, Aviation & Transport",2020-02-27,2020-02-27,2020-02-27,...,Case Outstanding,Gross,cumulative,2020.0,2026,2020.0,0.00,0,0,0.00
1534,SUN-MHU-23-0299,95CF5CF4-7298-40A3-BD7A-1034C0834D30,23S-0447A,reinsurance,selecta,Marine: H&M,"Marine, Aviation & Transport",2023-05-24,2023-05-24,2023-05-24,...,Case Outstanding,Gross,cumulative,2023.0,2026,2023.0,0.00,0,0,0.00
1535,SUN-MHU-23-0205,95CF5CF4-7298-40A3-BD7A-1034C0834D30,23S-0185A,reinsurance,sunshine_selecta,Marine: Yacht,"Marine, Aviation & Transport",2023-04-18,2023-04-06,2023-06-06,...,Case Outstanding,Gross,cumulative,2023.0,2026,2023.0,0.00,0,0,0.00
1536,SUN-MHU-23-0159,95CF5CF4-7298-40A3-BD7A-1034C0834D30,23S-0052A,reinsurance,sunshine_selecta,Marine: Yacht,"Marine, Aviation & Transport",2023-06-15,2023-05-20,2023-06-15,...,Case Outstanding,Gross,cumulative,2023.0,2026,2023.0,0.00,0,0,0.00


In [23]:
gross_to_net_reported_ratio_df['gross_reported'] =  gross_to_net_reported_ratio_df['case_estimates'] + gross_to_net_reported_ratio_df['Loss+LAE paid']
gross_to_net_reported_ratio_df['net_reported'] = gross_to_net_reported_ratio_df['gross_reported'] - (os_df['case_estimate_recoveries']+ gross_to_net_reported_ratio_df['paid_recovered'])

KeyError: 'Loss+LAE paid'

In [ ]:
# Just to check gross to net ratios by claim, not for use
gross_to_net_reported_ratio_df['g2n_reported_per_claim'] = gross_to_net_reported_ratio_df['net_reported']/gross_to_net_reported_ratio_df['gross_reported']

In [ ]:
gross_to_net_reported_ratio = gross_to_net_reported_ratio_df.groupby(['uwy', 'lob'])['net_reported'].sum()/gross_to_net_reported_ratio_df.groupby(['uwy', 'lob'])['gross_reported'].sum()

In [ ]:
gross_to_net_reported_ratio.reset_index()

In [ ]:
#Remeber this is the net/gross
gross_to_net_reported_ratio.reset_index().pivot(index='uwy', columns='lob', values=0).fillna(0).reset_index()

In [ ]:
#G2N ratios - user driven, to be applied on the Unpaid and future claims excluding the case estimates
g2n = {
    "Marine": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 1, 2023: 1, 2024: 1, 2025: 0.5,
    },
    "Aviation": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 1, 2023: 1, 2024: 1, 2025: 1,
    },
    "Liability": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 1, 2023: 1, 2024: 1, 2025: 1,
    },
    "Financial": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 0.6, 2023: 1, 2024: 1, 2025: 1,
    },
    "Misc": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 1, 2023: 1, 2024: 1, 2025: 1,
    },
    "Personal": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 1, 2023: 1, 2024: 1, 2025: 1,
    },
    "Property": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 1, 2023: 1, 2024: 1, 2025: 1,
    },
    "Treaty": {
        2014: 1, 2015: 1, 2016: 1, 2017: 1,
        2018: 1, 2019: 1, 2020: 1, 2021: 1,
        2022: 1, 2023: 1, 2024: 1, 2025: 1,
    },
}



In [ ]:
test_g2n = pd.DataFrame(g2n).stack().reset_index().rename(columns={'level_0': 'uwy', 'level_1': 'lob', 0: 'g2n_ratio'})

### Save data to csv to push to notebooks..

In [ ]:
import pandas as pd

g2n_df = (
    pd.DataFrame(g2n)
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'uwy', 'level_1': 'lob', 0: 'g2n_ratio'})
)
g2n_df['uwy'] = g2n_df['uwy'].astype(int)
g2n_df

In [ ]:
g2n_df.to_parquet(REPO_ROOT / 'data' / 'db' / 'g2n_ratios.parquet', index=False)

In [ ]:
os_2025 = os_df.groupby(['val_date', 'data_type', 'lob','gross_net_re', 'cum_inc', 'origin_period', 'dev_period',])[['case_estimates','net_ri_case_estimates']].sum().reset_index()

In [ ]:
#As we don't need net case estimates for the triangles, we are going to send it straight to the notebooks, summarised by lob and uwy. Remember as this is IFRS17, we present everything on gross and reinsurance not gross and net!
ri_os_2025 = os_df.groupby(['lob', 'origin_period'])['recoveries'].sum().reset_index()
ri_os_2025.rename(columns={'origin_period':'uwy'}, inplace=True)


In [ ]:
ri_os_2025

In [ ]:
os_2025.to_csv(REPO_ROOT /'data'/'db'/'case_estimates_2025.csv', index=False)
ri_os_2025.to_parquet(REPO_ROOT / 'data' / 'exhibits' / 'recoveries_2025.parquet', index=False)

# Save this to the db, under case_estimates

### ALAE Calcs
NOTE TO SELF: WE MAY BE ABLE TO INCLUDE AN ALAE ASSUMPTION, AS WE HAVE ALAE SEPARATED FROM LOSS PAID..

In [ ]:
alae_calcs = os_df[['uwy', 'lob', 'Loss adjustment expenses, Amount Outstanding,USD','Liability for incurred claims, Amount Outstanding,USD' ]].copy()

alae_calcs = alae_calcs.rename(columns={
    'uwy': 'uwy',
    'lob': 'lob',
    'Loss adjustment expenses, Amount Outstanding,USD': 'alae',
    'Liability for incurred claims, Amount Outstanding,USD': 'case_estimates'
})

In [ ]:
alae_calcs['alae'] = pd.to_numeric(alae_calcs['alae'], errors='coerce')
alae_calcs['case_estimates'] = pd.to_numeric(alae_calcs['case_estimates'], errors='coerce')
alae_calcs.groupby(['uwy', 'lob'])[['alae', 'case_estimates']].sum()

alae_calcs['alae_perc'] = (alae_calcs['alae']/alae_calcs['case_estimates']).fillna(0)

In [ ]:
alae_calcs.groupby(['uwy','lob'])[['alae','case_estimates']].sum().reset_index()